In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from openai import OpenAI
openai_client = OpenAI()

# Retrieval

In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()


In [6]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 482},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 161},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 398},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 73},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [7]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"{url_prefix}/{course["path"]}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [8]:
documents[10]

{'id': '2b5ff70c77',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Do I need to enroll in the course before submitting homework?',
 'answer': 'No enrollment is required to submit homework. Just log into the homework form when it opens. The Airtable registration you may see is only for announcements; actual submissions are made on the course platform forms and via your GitHub as specified in the homework guidelines.'}

In [9]:
from minsearch import Index

index = Index(
    text_fields = ["question", "section", "answer"],
    keyword_fields = ["course"]
)

index.fit(documents)

In [10]:
question = "I just discovered the course. Can I still join?"
search_results = index.search(question, filter_dict={"course": "llm-zoomcamp"}, num_results=5)

In [11]:
boosted_search_results = index.search(question, 
                              boost_dict={"question": 2.0, "section": 0.5}, # question field is 2x more important
                              filter_dict={"course": "llm-zoomcamp"}, 
                              num_results=5)

In [12]:
def search(question, course="llm-zoomcamp"):
    return index.search(question, 
                        boost_dict={"question": 2.0, "section": 0.5}, 
                        filter_dict={"course": course}, 
                        num_results=5
                        )

In [13]:
search_res = search(question)

# Building Prompt

In [14]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context, respond with 'I don't know.'
"""

In [16]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [17]:
context = build_context(search_res)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expec

In [18]:
USER_PROMPT_TEMPLATE = f"""
Question:
{question}

Context:
{context}
"""

In [19]:
USER_PROMPT_TEMPLATE.format(question=question, context=context)

"\nQuestion:\nI just discovered the course. Can I still join?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: The homework submission form is still open even though the deadline has passed — can I still submit?\nA: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.\n\nGeneral Course-Related Questions\nQ: I missed the first homework - can I still get a certificate?\nA: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.\n\nGeneral Cours

In [20]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

In [21]:
prompt = build_prompt(question, search_res)

In [22]:
print(prompt)

Question:
I just discovered the course. Can I still join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: The homework submission form is still open even though the deadline has passed — can I still submit?
A: Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit.

General Course-Related Questions
Q: I missed the first homework - can I still get a certificate?
A: Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

General Course-Related Questions

# Adding LLM

In [24]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)
response.output_text

'Yes. You can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [25]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0f12f3b9fb5c63a4006aee3b81219887d2a969b6a0e461a485",
  "access_programs": null,
  "created_at": 1793997697.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_0f12f3b9fb5c63a4006aee3b81ce2887d29590436cd07eb80f",
      "content": [
        {
          "annotations": [],
          "text": "Yes. You can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1793997698.0,
  "conversation": null,
  "image": {
    "context_compres

In [26]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
    model=model,
    input=user_prompt
    )

    return response.output_text